# Replikasi Eksperimen Giudici et al. (2020)
## Network Models to Improve Automated Cryptocurrency Portfolio Management

Notebook ini membandingkan strategi portofolio berikut:
1. **EW** — Equally Weighted (1/N)
2. **CM** — Classical Markowitz
3. **GM** — Glasso Markowitz
4. **NW (γ=0)** — Network Markowitz (RMT+MST murni)
5. **NW (γ=1.0)** — Network Markowitz + penalti sentralitas eigenvector
6. **AGGP v3** — Adaptive Graph-Gated Portfolio (Model Usulan)

---
## Sel 1 — Persiapan Library

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import networkx as nx
from scipy.optimize import minimize
from scipy.sparse.csgraph import minimum_spanning_tree
from scipy.linalg import eigh
from sklearn.covariance import GraphicalLassoCV
import warnings
warnings.filterwarnings('ignore')

plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette("husl")

print("Libraries imported successfully!")

---
## Sel 2 — Memuat Data

Memuat data return dan harga dari file Excel. Fokus pada dataset return harian untuk 10 aset kripto utama dalam periode **14 September 2017 – 17 Oktober 2019**.

In [ ]:
# Load data
excel_file = 'crypto_data_real.xlsx'
df_returns = pd.read_excel(excel_file, sheet_name='Returns', index_col=0)
df_prices  = pd.read_excel(excel_file, sheet_name='Prices',  index_col=0)

crypto_names = df_returns.columns.tolist()
n_assets     = len(crypto_names)

print(f"Data loaded: {df_returns.shape[0]} days, {n_assets} assets")
print(f"Period: {df_returns.index[0]} to {df_returns.index[-1]}")
print(f"Assets: {crypto_names}")

---
## Sel 3 — Statistik Ringkasan (TABLE 1)

Statistik deskriptif mencakup Mean, Standar Deviasi, Kurtosis, dan Skewness sesuai standar pelaporan literatur keuangan.

In [ ]:
# Create summary statistics table
summary_stats = pd.DataFrame({
    'Mean':     df_returns.mean(),
    'Std':      df_returns.std(),
    'Kurtosis': df_returns.kurt(),
    'Skewness': df_returns.skew()
})

print("TABLE 1 | Summary statistics.")
print(summary_stats.round(4))

---
## Sel 4 — Visualisasi Harga Ternormalisasi (Figure 1 & 2)

Harga awal setiap aset diatur menjadi **100** pada tanggal **7 Januari 2018**.

In [ ]:
# --- Figure 1: BTC, ETH, USDT, BCH, LTC ---
assets_1  = ['BTC', 'ETH', 'USDT', 'BCH', 'LTC']
df_norm_1 = (df_prices.loc['2018-01-07':, assets_1] / df_prices.loc['2018-01-07', assets_1]) * 100
colors_1  = ['black', 'red', 'green', 'blue', 'cyan']

plt.figure(figsize=(12, 5))
for i, col in enumerate(df_norm_1.columns):
    plt.plot(df_norm_1.index, df_norm_1[col], label=col, color=colors_1[i])
plt.title('Figure 1 | Normalized Price Series I (BTC, ETH, USDT, BCH, LTC)')
plt.ylabel('Normalized Price (base=100)')
plt.legend()
plt.tight_layout()
plt.show()

# --- Figure 2: XRP, BNB, EOS, XLM, TRX ---
assets_2  = ['XRP', 'BNB', 'EOS', 'XLM', 'TRX']
df_norm_2 = (df_prices.loc['2018-01-07':, assets_2] / df_prices.loc['2018-01-07', assets_2]) * 100
colors_2  = ['magenta', 'gold', 'lightgrey', 'black', 'red']

plt.figure(figsize=(12, 5))
for i, col in enumerate(df_norm_2.columns):
    plt.plot(df_norm_2.index, df_norm_2[col], label=col, color=colors_2[i])
plt.title('Figure 2 | Normalized Price Series II (XRP, BNB, EOS, XLM, TRX)')
plt.ylabel('Normalized Price (base=100)')
plt.legend()
plt.tight_layout()
plt.show()

---
## Sel 5 — Visualisasi Minimum Spanning Tree (Figure 3 & 4)

MST menunjukkan struktur jaringan aset kripto selama periode **gelembung spekulatif** (Sep 2017 – Jan 2018) dan periode **stabil** (Jun 2019 – Okt 2019).

In [ ]:
# --- Figure 3 | MST September 2017 - January 2018 ---
df_bubble   = df_returns.loc['2017-09-14':'2018-01-31']
corr_bubble = df_bubble.corr()
dist_bubble = np.sqrt(2 * (1 - corr_bubble))

G          = nx.from_pandas_adjacency(dist_bubble)
mst_bubble = nx.minimum_spanning_tree(G, weight='weight')

plt.figure(figsize=(10, 8))
pos = nx.spring_layout(mst_bubble, seed=42)
nx.draw(mst_bubble, pos, with_labels=True, node_color='orange',
        node_size=1500, edge_color='black', linewidths=1.5,
        font_size=10, font_weight='bold')
plt.title('Figure 3 | MST September 2017 - January 2018', fontsize=12)
plt.show()

# --- Figure 4 | MST June 2019 - October 2019 ---
df_stable   = df_returns.loc['2019-06-01':'2019-10-17']
corr_stable = df_stable.corr()
dist_stable = np.sqrt(2 * (1 - corr_stable))

G2         = nx.from_pandas_adjacency(dist_stable)
mst_stable = nx.minimum_spanning_tree(G2, weight='weight')

plt.figure(figsize=(10, 8))
pos2 = nx.spring_layout(mst_stable, seed=42)
nx.draw(mst_stable, pos2, with_labels=True, node_color='orange',
        node_size=1500, edge_color='black', linewidths=1.5,
        font_size=10, font_weight='bold')
plt.title('Figure 4 | MST June 2019 - October 2019', fontsize=12)
plt.show()

---
## Sel 6 — Fungsi Pembantu (Helper Functions)

Implementasi:
- **`apply_rmt_filter`**: Filter noise matriks korelasi via *Random Matrix Theory* (Marcenko-Pastur)
- **`build_mst`**: Konversi korelasi ke matriks jarak untuk MST
- **`compute_eigenvector_centrality`**: Hitung sentralitas eigenvector tiap aset
- **`calculate_var`**, **`calculate_rachev_ratio`**, **`calculate_max_drawdown`**: Metrik risiko

In [ ]:
def apply_rmt_filter(returns_data):
    """Filter noise dari matriks korelasi menggunakan Random Matrix Theory (Marcenko-Pastur)."""
    if isinstance(returns_data, pd.DataFrame):
        data = returns_data.values
    else:
        data = returns_data
    T, N = data.shape
    Q    = T / N
    C    = np.corrcoef(data.T)

    eigenvalues, eigenvectors = eigh(C)
    eigenvalues  = eigenvalues[::-1]
    eigenvectors = eigenvectors[:, ::-1]

    lambda_plus      = 1 + (1/Q) + 2*np.sqrt(1/Q)   # Marcenko-Pastur upper bound
    significant_mask = eigenvalues > lambda_plus
    Lambda_filtered  = np.diag(np.where(significant_mask, eigenvalues, 0))
    C_filtered       = eigenvectors @ Lambda_filtered @ eigenvectors.T
    return C_filtered


def build_mst(correlation_matrix):
    """Konversi matriks korelasi ke matriks jarak untuk konstruksi MST."""
    distance_matrix = np.sqrt(2 - 2*correlation_matrix)
    np.fill_diagonal(distance_matrix, 0)
    return distance_matrix


def compute_eigenvector_centrality(distance_matrix):
    """Hitung sentralitas eigenvector dari matriks jarak."""
    adjacency = 1 / (distance_matrix + 1e-8)
    np.fill_diagonal(adjacency, 0)
    eigenvalues, eigenvectors = eigh(adjacency)
    principal_eigenvector = np.abs(eigenvectors[:, -1])
    centrality = principal_eigenvector / principal_eigenvector.sum()
    return centrality


def calculate_var(returns, confidence=0.95):
    """Value at Risk pada tingkat kepercayaan tertentu."""
    return np.percentile(returns, (1-confidence)*100)


def calculate_rachev_ratio(returns, alpha=0.10):
    """Rachev Ratio = CVaR_upper(alpha) / CVaR_lower(alpha)."""
    threshold_upper = np.percentile(returns, (1-alpha)*100)
    threshold_lower = np.percentile(returns, alpha*100)
    cvar_upper = returns[returns >= threshold_upper].mean()
    cvar_lower = abs(returns[returns <= threshold_lower].mean())
    return cvar_upper / cvar_lower if cvar_lower > 0 else 0


def calculate_max_drawdown(cumulative_returns):
    """Maximum Drawdown dari seri cumulative returns."""
    running_max = np.maximum.accumulate(cumulative_returns)
    drawdown    = (cumulative_returns - running_max) / running_max
    return drawdown.min()


print("Helper functions defined successfully!")

---
## Sel New — Analisis Dinamika MST (Figure 5)

Analisis *rolling window* untuk menghitung **max link distance** dan **koefisien residuality** MST sepanjang waktu.

In [ ]:
def calculate_rolling_mst_metrics(returns_df, window=120):
    """Hitung dinamika MST dengan rolling window."""
    dates         = returns_df.index[window:]
    max_links     = []
    residualities = []

    for i in range(window, len(returns_df)):
        window_data  = returns_df.iloc[i - window:i]
        corr_f       = apply_rmt_filter(window_data)
        mst_weights  = build_mst(corr_f)
        max_links.append(np.max(mst_weights))
        residualities.append(np.sum(mst_weights) / (returns_df.shape[1] - 1))

    return pd.DataFrame(
        {'Max Link': max_links, 'Residuality': residualities},
        index=dates
    )


mst_dyn = calculate_rolling_mst_metrics(df_returns)

# Visualisasi dengan dual axis
fig, ax1 = plt.subplots(figsize=(12, 7))
ax1.plot(mst_dyn.index, mst_dyn['Max Link'], color='black', label='Max Link')
ax1.set_xlabel('Date')
ax1.set_ylabel('Max Link Distance', color='black')
ax1.tick_params(axis='y', labelcolor='black')

ax2 = ax1.twinx()
ax2.plot(mst_dyn.index, mst_dyn['Residuality'], color='red', label='Residuality')
ax2.set_ylabel('Residuality', color='red')
ax2.tick_params(axis='y', labelcolor='red')

lines1, labels1 = ax1.get_legend_handles_labels()
lines2, labels2 = ax2.get_legend_handles_labels()
ax1.legend(lines1 + lines2, labels1 + labels2, loc='upper right')

plt.title('Figure 5 | MST Dynamics: Max Link Distance & Residuality')
plt.tight_layout()
plt.show()

---
## Sel 7 — Implementasi Strategi Portofolio

### Perbaikan Kritis AGGP v3 terhadap AGGP v2:

| Fix | Masalah | Solusi |
|-----|---------|--------|
| **Fix A** | Sensor `avg_corr` dari `Pf` mentah → nilai tidak terkontrol (bisa >10) | Hitung **korelasi parsial proper**: ρ_ij = −P_ij / √(P_ii · P_jj), dijamin ∈ [0,1] |
| **Fix B** | `gamma_t` hampir selalu di batas bawah 0.1 | Dengan sensor [0,1], rumus `gamma_t = 1 − avg_pcorr` bekerja natural |
| **Fix C** | Tidak ada mekanisme *momentum switching* | Tambahkan **threshold eksplisit** τ untuk switching defensif/agresif |

In [ ]:
# ── Base Class ─────────────────────────────────────────────────────────────────
class PortfolioStrategy:
    def __init__(self, name):
        self.name            = name
        self.weights_history = []
        self.returns_history = []

    def get_weights(self, returns_data):
        raise NotImplementedError


# ── 1. Equally Weighted ────────────────────────────────────────────────────────
class EquallyWeighted(PortfolioStrategy):
    def get_weights(self, returns_data):
        n = returns_data.shape[1]
        return np.ones(n) / n


# ── 2. Classical Markowitz ─────────────────────────────────────────────────────
class ClassicalMarkowitz(PortfolioStrategy):
    """Optimasi Mean-Variance tradisional (minimasi varians portofolio)."""
    def get_weights(self, returns_data):
        n  = returns_data.shape[1]
        mu = returns_data.mean().values
        S  = returns_data.cov().values
        objective   = lambda w: w @ S @ w
        constraints = [
            {'type': 'eq',   'fun': lambda w: np.sum(w) - 1},
            {'type': 'ineq', 'fun': lambda w: w @ mu - mu.mean()}
        ]
        res = minimize(objective, np.ones(n) / n,
                       method='SLSQP', bounds=[(0, 1)] * n,
                       constraints=constraints)
        return res.x if res.success else np.ones(n) / n


# ── 3. Glasso Markowitz ────────────────────────────────────────────────────────
class GlassoMarkowitz(PortfolioStrategy):
    """Markowitz dengan matriks kovarians yang diestimasi via Graphical Lasso."""
    def get_weights(self, returns_data):
        n  = returns_data.shape[1]
        mu = returns_data.mean().values
        try:
            glasso = GraphicalLassoCV()
            glasso.fit(returns_data.values)
            S = glasso.covariance_
        except Exception:
            S = returns_data.cov().values
        objective   = lambda w: w @ S @ w
        constraints = [
            {'type': 'eq',   'fun': lambda w: np.sum(w) - 1},
            {'type': 'ineq', 'fun': lambda w: w @ mu - mu.mean()}
        ]
        res = minimize(objective, np.ones(n) / n,
                       method='SLSQP', bounds=[(0, 1)] * n,
                       constraints=constraints)
        return res.x if res.success else np.ones(n) / n


# ── 4. Network Markowitz ───────────────────────────────────────────────────────
class NetworkMarkowitz(PortfolioStrategy):
    """Network Markowitz: RMT + MST + penalti sentralitas eigenvector."""
    def __init__(self, name="Network Markowitz", gamma=0):
        super().__init__(name)
        self.gamma = gamma

    def get_weights(self, returns_data):
        n   = returns_data.shape[1]
        mu  = returns_data.mean().values
        sig = returns_data.std().values
        Cf   = apply_rmt_filter(returns_data)
        dist = build_mst(Cf)
        cent = compute_eigenvector_centrality(dist)
        Sf   = np.outer(sig, sig) * Cf
        objective   = lambda w: w @ Sf @ w + self.gamma * np.sum(cent * w)
        constraints = [
            {'type': 'eq',   'fun': lambda w: np.sum(w) - 1},
            {'type': 'ineq', 'fun': lambda w: w @ mu - mu.mean()}
        ]
        res = minimize(objective, np.ones(n) / n,
                       method='SLSQP', bounds=[(0, 1)] * n,
                       constraints=constraints)
        return res.x if res.success else np.ones(n) / n


# ── 5. Adaptive Graph-Gated Portfolio (AGGP v3) ────────────────────────────────
class AdaptiveGraphPortfolio(PortfolioStrategy):
    """
    AGGP v3: Sensor kondisi pasar menggunakan PARTIAL CORRELATION yang proper,
    dijamin berada di rentang [0, 1].

    Sensor: avg_pcorr = mean(|partial_corr_ij|), i != j
    Partial Corr: rho_ij = -P[i,j] / sqrt(P[i,i] * P[j,j])

    Mekanisme adaptasi gamma_t (FIX A + FIX B + FIX C):
      - avg_pcorr >= stress_threshold -> gamma_t = gamma_min  (defensif)
      - avg_pcorr <  stress_threshold -> soft blending agresif ke NW gamma=1.0
    """
    def __init__(self, name="AGGP v3", stress_threshold=0.25,
                 gamma_min=0.1, gamma_max=0.9):
        super().__init__(name)
        self.stress_threshold = stress_threshold
        self.gamma_min        = gamma_min
        self.gamma_max        = gamma_max

    def get_weights(self, returns_data):
        n   = returns_data.shape[1]
        mu  = returns_data.mean().values
        sig = returns_data.std().values

        # --- Estimasi kovarians & presisi via Graphical Lasso ---
        try:
            glasso = GraphicalLassoCV(cv=3)
            glasso.fit(returns_data.values)
            Sf = glasso.covariance_
            Pf = glasso.precision_
        except Exception:
            Cf = apply_rmt_filter(returns_data)
            Sf = np.outer(sig, sig) * Cf
            Pf = np.linalg.pinv(Sf)

        # --- FIX A: Partial Correlation proper (dijamin di [0, 1]) ---
        # rho_partial_ij = -P[i,j] / sqrt(P[i,i] * P[j,j])
        D_inv     = 1.0 / np.sqrt(np.diag(Pf))
        P_corr    = -Pf * np.outer(D_inv, D_inv)   # partial corr matrix
        np.fill_diagonal(P_corr, 0.0)
        mask      = ~np.eye(n, dtype=bool)
        avg_pcorr = np.clip(np.mean(np.abs(P_corr[mask])), 0.0, 1.0)

        # --- FIX B + FIX C: Threshold switching + soft blending ---
        if avg_pcorr >= self.stress_threshold:
            # Pasar STRES: lindungi modal, minimasi paparan ke Network
            gamma_t = self.gamma_min
        else:
            # Pasar NORMAL: manfaatkan struktur jaringan secara agresif
            ratio   = avg_pcorr / self.stress_threshold  # [0, 1)
            gamma_t = self.gamma_max * (1.0 - ratio)     # linear dari max ke 0
            gamma_t = np.clip(gamma_t, self.gamma_min, self.gamma_max)

        # --- KOMPONEN NETWORK: NW gamma=1.0 ---
        Cf_rmt = apply_rmt_filter(returns_data)
        dist   = build_mst(Cf_rmt)
        cent   = compute_eigenvector_centrality(dist)
        Sf_nw  = np.outer(sig, sig) * Cf_rmt
        obj_nw  = lambda w: w @ Sf_nw @ w + 1.0 * np.sum(cent * w)
        cons_nw = [
            {'type': 'eq',   'fun': lambda w: np.sum(w) - 1},
            {'type': 'ineq', 'fun': lambda w: w @ mu - mu.mean()}
        ]
        res_nw  = minimize(obj_nw, np.ones(n) / n, method='SLSQP',
                           bounds=[(0, 1)] * n, constraints=cons_nw)
        w_net   = res_nw.x if res_nw.success else np.ones(n) / n

        # --- KOMPONEN MARKOWITZ: GMV via Glasso + return constraint ---
        obj_mko  = lambda w: w @ Sf @ w
        cons_mko = [
            {'type': 'eq',   'fun': lambda w: np.sum(w) - 1},
            {'type': 'ineq', 'fun': lambda w: w @ mu - mu.mean()}
        ]
        res_mko  = minimize(obj_mko, np.ones(n) / n, method='SLSQP',
                            bounds=[(0, 1)] * n, constraints=cons_mko)
        w_mko    = res_mko.x if res_mko.success else np.ones(n) / n

        # --- BLENDING ADAPTIF ---
        return (1.0 - gamma_t) * w_mko + gamma_t * w_net


print("All strategy classes defined successfully!")

---
## Sel 8 — Framework Backtesting

Pengujian menggunakan **rolling window** dengan:
- `window_size = 120` hari training
- `rebalance_freq = 7` hari
- `transaction_cost = 0.1%` (10 basis points)

In [ ]:
def backtest_strategy(strategy, df_returns, window_size=120,
                      rebalance_freq=7, transaction_cost=0.001):
    """Simulasi backtest dengan rolling window dan biaya transaksi."""
    portfolio_returns = []
    dates             = []

    for i in range(window_size, len(df_returns), rebalance_freq):
        train = df_returns.iloc[i - window_size:i]
        w     = strategy.get_weights(train)

        test_end  = min(i + rebalance_freq, len(df_returns))
        test_data = df_returns.iloc[i:test_end]

        for j in range(len(test_data)):
            daily_ret = np.dot(w, test_data.iloc[j].values)
            if j == 0 and len(portfolio_returns) > 0:
                daily_ret -= transaction_cost
            portfolio_returns.append(daily_ret)
            dates.append(test_data.index[j])

    res_df = pd.DataFrame({'date': dates, 'return': portfolio_returns})
    res_df['cumulative_return'] = (1 + res_df['return']).cumprod()

    return {
        'strategy':           strategy.name,
        'returns':            np.array(portfolio_returns),
        'cumulative_returns': res_df['cumulative_return'].values,
        'results_df':         res_df
    }

print("Backtest function defined successfully!")

---
## Sel 9 — Eksekusi Backtesting

In [ ]:
# --- Inisialisasi strategi ---
strategies = [
    EquallyWeighted("EW"),
    ClassicalMarkowitz("CM"),
    GlassoMarkowitz("GM"),
    NetworkMarkowitz("NW (gamma=0)",   gamma=0),
    NetworkMarkowitz("NW (gamma=1.0)", gamma=1.0),
    AdaptiveGraphPortfolio("AGGP v3",
                           stress_threshold=0.25,
                           gamma_min=0.1,
                           gamma_max=0.9),
]

# --- Eksekusi backtest ---
results = {}
for strat in strategies:
    print(f"Backtesting: {strat.name}...")
    results[strat.name] = backtest_strategy(strat, df_returns)
    final = (results[strat.name]['cumulative_returns'][-1] - 1) * 100
    print(f"  -> Done. Final cumulative return: {final:.2f}%")

print("\nAll backtests completed!")

---
## Sel 10 — Analisis Performa Periodik (TABLE 2)

Perbandingan performa kumulatif yang disampel setiap **4 bulan** (Januari, Mei, September).

In [ ]:
target_dates = [
    '2018-01-31', '2018-05-31', '2018-09-30',
    '2019-01-31', '2019-05-31', '2019-09-30'
]

table2_rows = []
for strat_name, res in results.items():
    df_res = res['results_df'].set_index('date')
    row    = {'Strategy': strat_name}
    for td in target_dates:
        idx     = df_res.index.searchsorted(pd.Timestamp(td))
        idx     = min(idx, len(df_res) - 1)
        cum_ret = (df_res['cumulative_return'].iloc[idx] - 1) * 100
        row[td] = round(cum_ret, 2)
    table2_rows.append(row)

table2 = pd.DataFrame(table2_rows).set_index('Strategy')
table2.columns = ['Jan-2018', 'May-2018', 'Sep-2018', 'Jan-2019', 'May-2019', 'Sep-2019']

print("TABLE 2 | Cumulative Profits and Losses (%)")
print(table2.to_string())

---
## Sel 11 — Visualisasi Performa Kumulatif (FIGURE 6)

Evolusi nilai portofolio dengan asumsi **investasi awal 100 USD**.

In [ ]:
plt.figure(figsize=(14, 7))

for name, res in results.items():
    plt.plot(res['results_df']['date'],
             res['cumulative_returns'] * 100,
             label=name)

plt.title('FIGURE 6 | Performances of Different Portfolio Strategies')
plt.xlabel('Date')
plt.ylabel('Portfolio Value (USD, initial = 100)')
plt.legend(bbox_to_anchor=(1.02, 1), loc='upper left')
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

---
## Sel 12 — Analisis Risiko Periodik (TABLE 3: VaR 95%)

**Value at Risk** dengan tingkat kepercayaan 95%, dihitung pada jendela 4 bulan terakhir di setiap titik sampling. Nilai absolut dikali faktor skala 100.

In [ ]:
def get_returns_segment(res_df, start_date, end_date):
    """Ambil segmen return pada rentang tanggal tertentu."""
    mask = (res_df['date'] >= pd.Timestamp(start_date)) & \
           (res_df['date'] <= pd.Timestamp(end_date))
    return res_df.loc[mask, 'return'].values


# Batas 4 bulan sebelum setiap target date
var_windows = [
    ('2017-09-30', '2018-01-31', 'Jan-2018'),
    ('2018-01-31', '2018-05-31', 'May-2018'),
    ('2018-05-31', '2018-09-30', 'Sep-2018'),
    ('2018-09-30', '2019-01-31', 'Jan-2019'),
    ('2019-01-31', '2019-05-31', 'May-2019'),
    ('2019-05-31', '2019-09-30', 'Sep-2019'),
]

table3_rows = []
for strat_name, res in results.items():
    row = {'Strategy': strat_name}
    for start, end, label in var_windows:
        seg = get_returns_segment(res['results_df'], start, end)
        if len(seg) > 0:
            v = abs(calculate_var(seg, 0.95)) * 100
        else:
            v = np.nan
        row[label] = round(v, 4)
    table3_rows.append(row)

table3 = pd.DataFrame(table3_rows).set_index('Strategy')
print("TABLE 3 | VaR 95% (4-month windows, scale x100)")
print(table3.to_string())

---
## Sel 13 — Analisis Risk-Adjusted Return (TABLE 4: Sharpe Ratio)

In [ ]:
table4_rows = []
for strat_name, res in results.items():
    row = {'Strategy': strat_name}
    for start, end, label in var_windows:
        seg = get_returns_segment(res['results_df'], start, end)
        if len(seg) > 1 and np.std(seg) > 0:
            sr = np.mean(seg) / np.std(seg)
        else:
            sr = np.nan
        row[label] = round(sr, 4)
    table4_rows.append(row)

table4 = pd.DataFrame(table4_rows).set_index('Strategy')
print("TABLE 4 | Sharpe Ratio (4-month windows)")
print(table4.to_string())

---
## Sel 14 — Analisis Tail-Risk (TABLE 5: Rachev Ratio)

**Rachev Ratio** = CVaR_upper(10%) / CVaR_lower(10%). Mengukur asimetri distribusi imbal hasil pada ekor (*tails*).

In [ ]:
table5_rows = []
for strat_name, res in results.items():
    row = {'Strategy': strat_name}
    for start, end, label in var_windows:
        seg = get_returns_segment(res['results_df'], start, end)
        if len(seg) > 0:
            rr = calculate_rachev_ratio(seg, alpha=0.10)
        else:
            rr = np.nan
        row[label] = round(rr, 4)
    table5_rows.append(row)

table5 = pd.DataFrame(table5_rows).set_index('Strategy')
print("TABLE 5 | Rachev Ratio (4-month windows, alpha=10%)")
print(table5.to_string())

---
## Sel 15 — Analisis Fase Pasar (TABLE 6: Market Phase Analysis)

Pemetaan performa ke tiga fase pasar:
- **Bearish**: Jan 2018 – Mar 2019
- **Recovery**: Apr 2019 – Jun 2019
- **Stable/Sideways**: Jul 2019 – Okt 2019

Hipotesis: AGGP v3 seharusnya mengalahkan NW(γ=1.0) pada fase Bearish (mode defensif aktif) dan bersaing setara/lebih baik pada fase Recovery/Stable.

In [ ]:
# Definisi Fase Pasar
fase_pasar = {
    'Bearish':   ('2018-01-01', '2019-03-31'),
    'Recovery':  ('2019-04-01', '2019-06-30'),
    'Stable':    ('2019-07-01', '2019-10-17'),
}

# --- Panel A: Sharpe Ratio per Fase ---
sharpe_rows = []
rachev_rows = []

for strat_name, res in results.items():
    sr_row = {'Strategy': strat_name}
    rr_row = {'Strategy': strat_name}
    for fase, (start, end) in fase_pasar.items():
        seg = get_returns_segment(res['results_df'], start, end)
        if len(seg) > 1 and np.std(seg) > 0:
            sr_row[fase] = round(np.mean(seg) / np.std(seg), 4)
        else:
            sr_row[fase] = np.nan
        if len(seg) > 0:
            rr_row[fase] = round(calculate_rachev_ratio(seg, alpha=0.10), 4)
        else:
            rr_row[fase] = np.nan
    sharpe_rows.append(sr_row)
    rachev_rows.append(rr_row)

table6_sharpe = pd.DataFrame(sharpe_rows).set_index('Strategy')
table6_rachev = pd.DataFrame(rachev_rows).set_index('Strategy')

print("TABLE 6 | Market Phase Performance")
print("\nPanel A: Sharpe Ratio")
print(table6_sharpe.to_string())
print("\nPanel B: Rachev Ratio (alpha=10%)")
print(table6_rachev.to_string())

---
## Ringkasan Hasil

Sel ini menampilkan ringkasan performa akhir seluruh strategi dalam satu tabel.

In [ ]:
summary_rows = []
for strat_name, res in results.items():
    rets = res['returns']
    cum  = res['cumulative_returns']
    summary_rows.append({
        'Strategy':          strat_name,
        'Final Return (%)':  round((cum[-1] - 1) * 100, 2),
        'Ann. Sharpe':       round(np.mean(rets) / np.std(rets) * np.sqrt(252), 4),
        'Max Drawdown (%)':  round(calculate_max_drawdown(cum) * 100, 2),
        'VaR 95% (x100)':   round(abs(calculate_var(rets, 0.95)) * 100, 4),
        'Rachev (10%)':      round(calculate_rachev_ratio(rets, 0.10), 4),
    })

summary_df = pd.DataFrame(summary_rows).set_index('Strategy')
print("=" * 70)
print("RINGKASAN PERFORMA KESELURUHAN")
print("=" * 70)
print(summary_df.to_string())